# 数学建模工作台 (Mathematical Modeling Workbench)

本 notebook 与 `math-modeling/MATH_PROTOCOL.md` 配合使用。

**使用方法**
- 用 `math_code` 工具或 Jupyter 执行 Python 代码。
- 推导、数值格式、灵敏度、验证与可视化都保存在此 notebook。
- 每个阶段前先阅读 `MATH_PROTOCOL.md`。

**推荐的建模流程**
1. 问题理解与目标
2. 假设与符号定义
3. 第一性原理建立方程
4. 解析/符号求解 (`sympy`)
5. 数值求解 (`numpy/scipy`, 显式/隐式差分)
6. 参数估计与拟合
7. 灵敏度与稳健性
8. 验证、误差分析与结果解释
9. 可视化与交付


In [1]:
# 基础环境检查
import os
import sys
import tempfile

# 避免 WSL 下 matplotlib 缓存目录不可写
os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp(prefix="mplconfig-"))

import numpy as np
import scipy
import sympy as sp
import matplotlib
matplotlib.use("Agg")
import pandas as pd

print(f"Python: {sys.version}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sympy: {sp.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"pandas: {pd.__version__}")


Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
numpy: 2.3.3
scipy: 1.16.2
sympy: 1.14.0
matplotlib: 3.10.6
pandas: 2.3.3


## 示例：一阶常微分方程隐式 Euler

**模型**：$y'(t) = -\lambda y(t)$，$y(0)=1$。

**隐式 Euler 格式**：
$$ y_{n+1} = y_n - \lambda \Delta t\, y_{n+1} \quad\Rightarrow\quad y_{n+1} = \frac{y_n}{1+\lambda\Delta t}. $$

**稳定性**：放大因子 $|1/(1+\lambda\Delta t)| < 1$ 对任意 $\Delta t>0$ 成立，因此是无条件稳定 (A-stable)。


In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

lam = 2.0
dt = 0.1
t_max = 5.0
n_steps = int(t_max / dt)

t = np.linspace(0, t_max, n_steps + 1)
y = np.empty_like(t)
y[0] = 1.0
for n in range(n_steps):
    y[n + 1] = y[n] / (1.0 + lam * dt)

# 解析解
y_exact = np.exp(-lam * t)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, y, "o-", label="Implicit Euler")
ax.plot(t, y_exact, "--", label="Exact")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Implicit Euler for y' = -lambda y")
ax.legend()
fig.savefig("implicit_euler_demo.png", dpi=150)
print("Saved:", __import__("os").path.abspath("implicit_euler_demo.png"))
print("max abs error:", np.max(np.abs(y - y_exact)))


Saved: \\wsl.localhost\Ubuntu\home\theresa\dsh-mathematical-modeling\math-modeling\implicit_euler_demo.png
max abs error: 0.03399813084501868


## 示例：符号推导

用 `sympy` 推导一阶线性 ODE 的通解，并检查隐式格式的稳定性表达式。


In [3]:
import sympy as sp

t = sp.symbols("t", positive=True)
y = sp.Function("y")
lam = sp.symbols("lambda", positive=True)
ode = sp.Eq(y(t).diff(t), -lam * y(t))
sol = sp.dsolve(ode, y(t))
print("ODE:", ode)
print("General solution:", sol)

# 隐式 Euler 放大因子
dt = sp.symbols("Delta_t", positive=True)
G_implicit = 1 / (1 + lam * dt)
G_explicit = 1 - lam * dt
print("Implicit Euler amplification factor:", G_implicit)
print("Explicit Euler amplification factor:", G_explicit)


ODE: Eq(Derivative(y(t), t), -lambda*y(t))
General solution: Eq(y(t), C1*exp(-lambda*t))
Implicit Euler amplification factor: 1/(Delta_t*lambda + 1)
Explicit Euler amplification factor: -Delta_t*lambda + 1
